# Predict GW4 and check against real results

Loads the GW4 feature DataFrame from `Feature_engineering.feature_builder`,
runs it through the trained xgboost model, and compares
`predicted_points` against the actual GW4 `total_points` already sitting
in `ml.player_gw_stats` — our first genuine out-of-sample accuracy check.

Read-only end to end: no writes to the database, no changes to `model.pkl`.

In [1]:
import sys
from pathlib import Path

# Run from Predict/, so cwd().parent is backend/ -- needed for Shared.db_utils,
# which replaced the five per-package db_utils.py copies.
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / "Feature_engineering"))

import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sqlalchemy import text

from Shared.db_utils import get_engine
from feature_builder import build_features, FEATURE_COLS

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

SEASON = "2025-26"
TARGET_GW = 4

engine = get_engine()


## 1. Build features for the target gameweek

In [2]:
features = build_features(engine, SEASON, TARGET_GW)
print("features shape:", features.shape)
features.head()

features shape: (841, 21)


,pts_rolling_3gw,pts_rolling_5gw,minutes_rolling_3gw,goals_rolling_5gw,assists_rolling_5gw,clean_sheets_rolling_5gw,saves_rolling_3gw,bonus_rolling_3gw,bps_rolling_3gw,ict_rolling_3gw,xg_rolling_5gw,xa_rolling_5gw,xgi_rolling_5gw,position_encoded,was_home,price_current,ownership_log,shots_per_90,shots_on_target_per_90,on_off_diff,gk_save_pct
player_id,,,,,,,,,,,,,,,,,,,,,
1,6.000000,6.000000,90.0,0.0,0.0,0.666667,3.333333,1.0,26.0,3.066667,0.00,0.006667,0.006667,0,True,5.5,14.693877,NaN,NaN,NaN,NaN
2,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.00,0.000000,0.000000,0,True,4.4,11.302414,NaN,NaN,NaN,NaN
3,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.00,0.000000,0.000000,0,True,4.0,10.977431,NaN,NaN,NaN,NaN
4,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.00,0.000000,0.000000,0,True,4.0,9.945397,NaN,NaN,NaN,NaN
5,4.666667,4.666667,90.0,0.0,0.0,0.666667,0.000000,0.0,19.0,2.300000,0.02,0.006667,0.026667,1,True,6.1,14.644821,NaN,NaN,NaN,NaN


## 2. Load the trained model

**Note:** the training notebook (`FPL_Model_1.ipynb`) saved the model two
ways: `pickle.dump(best_xgb, f)` -> `model.pkl`, and
`best_xgb.get_booster().save_model('model.json')` -> a native xgboost JSON
booster dump. The pickle is not portable across xgboost versions/builds --
it fails under this venv's xgboost 3.3.0 with `XGBoostError: input stream
corrupted` (confirmed identical, byte-for-byte, across three separate
re-saves/re-downloads of the .pkl). The JSON dump has no such issue, so
`backend/Data_ingestion/model.json` (feature names verified to match `feature_cols`
exactly) is the canonical model artifact for this project -- load that
directly rather than touching the pickle at all.

In [3]:
MODEL_JSON = Path.cwd().parent / "Data_ingestion" / "model.json"

booster = xgb.Booster()
booster.load_model(str(MODEL_JSON))

print("Model loaded from:", MODEL_JSON)
print("Booster feature_names match FEATURE_COLS order:", booster.feature_names == FEATURE_COLS)

Model loaded from: D:\Exp\backend\Data_ingestion\model.json
Booster feature_names match FEATURE_COLS order: True


## 3. Predict

In [4]:
dmatrix = xgb.DMatrix(features[FEATURE_COLS], feature_names=FEATURE_COLS, missing=np.nan)
predicted = booster.predict(dmatrix)

predictions = pd.DataFrame({"predicted_points": predicted}, index=features.index)
predictions.head()

,predicted_points
player_id,
1,5.755081
2,1.117470
3,1.038054
4,0.942236
5,5.874146


## 4. Pull real GW4 results and compare

In [5]:
actuals = pd.read_sql(
    text("""
        SELECT player_id, total_points AS actual_points
        FROM ml.player_gw_stats
        WHERE season = :season AND gameweek = :gw
    """),
    engine,
    params={"season": SEASON, "gw": TARGET_GW},
).set_index("player_id")

names = pd.read_sql(
    text("SELECT id AS player_id, web_name FROM ml.players WHERE season = :season"),
    engine,
    params={"season": SEASON},
).set_index("player_id")

compare = names.join(predictions).join(actuals, how="inner")
print("players with both a prediction and a real GW4 result:", len(compare))
compare.head()

players with both a prediction and a real GW4 result: 740


,web_name,predicted_points,actual_points
player_id,,,
1,Raya,5.755081,6
2,Arrizabalaga,1.117470,0
3,Hein,1.038054,0
4,Setford,0.942236,0
5,Gabriel,5.874146,9


## 5. Accuracy check

Compare against the validation metrics recorded in `model_metadata.json`
(xgboost_v1: RMSE 2.0156, MAE 0.9842) as a sanity bound -- this is a single
live gameweek, not the validation set, so some drift is expected, especially
given the ~5.9% of feature importance (FBref features) we can't populate
yet and the 129 players with no rolling history this gameweek.

In [6]:
compare_scored = compare.dropna(subset=["predicted_points", "actual_points"])
error = compare_scored["predicted_points"] - compare_scored["actual_points"]

mae = error.abs().mean()
rmse = np.sqrt((error ** 2).mean())
within_2 = (error.abs() <= 2).mean() * 100
within_4 = (error.abs() <= 4).mean() * 100

print(f"n = {len(compare_scored)}")
print(f"MAE:  {mae:.4f}   (val MAE was 0.9842)")
print(f"RMSE: {rmse:.4f}   (val RMSE was 2.0156)")
print(f"within 2 pts: {within_2:.1f}%   (val was 85.5%)")
print(f"within 4 pts: {within_4:.1f}%   (val was 94.3%)")

n = 740
MAE:  1.8538   (val MAE was 0.9842)
RMSE: 2.4964   (val RMSE was 2.0156)
within 2 pts: 70.4%   (val was 85.5%)
within 4 pts: 89.5%   (val was 94.3%)


### 5b. A real gotcha: rows with zero history

XGBoost's `missing=nan` handling routes an all-NaN row down a fixed
default branch at every split -- it is not a "give up, predict the
average" signal, it's just whichever direction each tree happened to
default to at training time. Players who joined after GW1-3 (transfers,
new signings) but did play in GW4 have every rolling/price/ownership
feature as NaN, and the model still emits a confident-looking, near-identical
prediction (~7.5 pts) for all of them regardless of who they are. Splitting
the error by whether a player had any history at all shows how much this
drags down the headline numbers above.

In [7]:
has_history = compare_scored.index.map(lambda pid: pd.notna(features.loc[pid, "pts_rolling_3gw"]))

for label, mask in [("has >=1 prior GW", has_history), ("zero prior history", ~has_history)]:
    sub = compare_scored[mask]
    err = sub["predicted_points"] - sub["actual_points"]
    print(f"{label:22s} n={len(sub):4d}  MAE={err.abs().mean():.4f}  RMSE={np.sqrt((err ** 2).mean()):.4f}")

has >=1 prior GW       n= 712  MAE=1.6781  RMSE=2.1836
zero prior history     n=  28  MAE=6.3218  RMSE=6.5923


## 6. Top 20 predicted, with actual GW4 result alongside

In [8]:
compare["error"] = compare["predicted_points"] - compare["actual_points"]
compare.sort_values("predicted_points", ascending=False).head(20)

,web_name,predicted_points,actual_points,error
player_id,,,,
703,Traoré,7.515514,0,7.515514
78,Lindelöf,7.515514,1,6.515514
653,Bakwa,7.515514,1,6.515514
652,Cuiabano,7.515514,0,7.515514
651,Savona,7.515514,0,7.515514
746,Xavi,7.515514,6,1.515514
701,Geertruida,7.515514,1,6.515514
572,Lammens,7.515514,0,7.515514
650,John,7.515514,0,7.515514


## 7. Biggest misses -- worth a look before trusting this further

In [9]:
compare_scored.assign(
    web_name=compare.loc[compare_scored.index, "web_name"],
    error=error,
).reindex(columns=["web_name", "predicted_points", "actual_points", "error"]).sort_values(
    "error", key=lambda s: s.abs(), ascending=False
).head(15)

,web_name,predicted_points,actual_points,error
player_id,,,,
25,Zubimendi,5.182041,16,-10.817959
513,Foden,2.219442,12,-9.780558
721,Van de Ven,5.438400,14,-8.561600
731,Bergvall,2.520509,11,-8.479491
703,Traoré,7.515514,0,7.515514
572,Lammens,7.515514,0,7.515514
650,John,7.515514,0,7.515514
651,Savona,7.515514,0,7.515514
652,Cuiabano,7.515514,0,7.515514


## 8. Confidence flags and tiers (`tier_builder.build_tiers`)

Two problems with using `predicted_points` directly, both visible above:
- The flat ~7.5 pt prediction for players with zero prior history (section
  5b) can outrank real in-form players in a naive "top predicted" sort.
- The model has no idea whether a player is actually going to play --
  `ml.players.status` does.

`build_tiers` only assigns a percentile tier (Elite/Strong/Average/Weak) to
players who are both known (`has_history`) and available (`status == 'a'`).
Everyone else gets a status label instead of a number -- read-only, no
writes, no changes to `feature_builder.py` or `model.json`.

In [10]:
from tier_builder import build_tiers

tiers = build_tiers(engine, SEASON, features, predictions)
tiers.head()

,player_id,web_name,predicted_points,has_history,availability_status,tier_or_label
0,1,Raya,5.755081,True,a,Elite
1,2,Arrizabalaga,1.117470,True,a,Average
2,3,Hein,1.038054,True,u,Doubtful/Injured/Unavailable
3,4,Setford,0.942236,True,a,Weak
4,5,Gabriel,5.874146,True,a,Elite


### 8a. Tier distribution

In [11]:
tiers["tier_or_label"].value_counts()

tier_or_label
Doubtful/Injured/Unavailable    278
Average                         178
New/Insufficient Data           117
Strong                          112
Weak                            111
Elite                            45
Name: count, dtype: int64

### 8b. The flat-prediction cluster now gets labeled, not ranked

Traoré, Cuiabano, Savona (and Hincapié, one of the zero-history players we
spot-checked against real FPL data earlier) all had the same ~7.5 pt flat
prediction. They should now show `"New/Insufficient Data"` instead of a
tier.

In [12]:
flat_prediction_players = [703, 652, 651, 35]  # Traore, Cuiabano, Savona, Hincapie
tiers[tiers["player_id"].isin(flat_prediction_players)]

,player_id,web_name,predicted_points,has_history,availability_status,tier_or_label
34,35,Hincapie,7.492132,False,a,New/Insufficient Data
650,651,Savona,7.515514,False,i,Doubtful/Injured/Unavailable
651,652,Cuiabano,7.515514,False,u,Doubtful/Injured/Unavailable
702,703,Traoré,7.515514,False,a,New/Insufficient Data


### 8c. Corrected top 10 -- tiered players only, ranked by predicted_points

Compare this against section 6's top 20 above, which was dominated by the
flat-prediction cluster. This one should read like real in-form players.

In [13]:
tiered_only = tiers[tiers["tier_or_label"].isin(["Elite", "Strong", "Average", "Weak"])]
tiered_only.sort_values("predicted_points", ascending=False).head(10)

,player_id,web_name,predicted_points,has_history,availability_status,tier_or_label
525,526,Haaland,7.411787,True,a,Elite
292,293,João Pedro,7.112670,True,a,Elite
471,472,M.Salah,6.682587,True,a,Elite
281,282,Neto,6.633997,True,a,Elite
308,309,Muñoz,6.519927,True,a,Elite
19,20,Rice,6.489242,True,a,Elite
552,553,B.Fernandes,6.226334,True,a,Elite
533,534,Mbeumo,6.211670,True,a,Elite
282,283,Enzo,6.142628,True,a,Elite
474,475,Gakpo,6.048794,True,a,Elite
